In [529]:
from sklearn.datasets import load_iris

from sklearn.model_selection import train_test_split

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, auc, confusion_matrix, f1_score, recall_score
from sklearn.svm import SVC

In [530]:
from collections import Counter
from nltk.util import ngrams

import numpy as np

In [536]:
from pathlib import Path
import numpy as np

from textfabric_utils import get_verses

In [532]:
# Try Fs("book@en").items() to get names of the book included in the text-fabric dataset
ot_train_books = {
    "Genesis": list(range(1,51)),
    "Exodus": list(range(1,22))
}

ot_test_books = {
    "Deuteronomy": list(range(1,21))
}

In [534]:
# Parse the dataset and get verses
# each verse in a tuple "(`verse reference`: str, `transliteration as a list of words`: list[str])"

ot_train_book_verses = get_verses(ot_train_books)
# print(ot_train_book_verses)

ot_test_book_verses = get_verses(ot_test_books)

**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
book,65,6566.69,100
chapter,1269,336.36,100
verse,31341,13.62,100
word,426835,1.00,100


Genesis
Exodus


**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
book,65,6566.69,100
chapter,1269,336.36,100
verse,31341,13.62,100
word,426835,1.00,100


Deuteronomy


In [483]:
nt_train_books = {
    "Matthew": list(range(1,31)),
    "Mark": list(range(1,17)),
    "Luke": list(range(1,24)),
    "John": list(range(1,22)),
}

nt_test_books = {
    "Acts": list(range(1,29))
}

In [484]:
nt_train_book_verses = get_verses(nt_train_books, target_fabric="etcbc/syrnt", ver="0.1")
# print(nt_train_book_verses)

nt_test_book_verses = get_verses(nt_test_books, target_fabric="etcbc/syrnt", ver="0.1")

**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
book,27,4060.74,100
chapter,260,421.69,100
lexeme,3038,36.09,100
verse,7957,13.78,100
word,109640,1.00,100


Matthew
Mark
Luke
John


**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
book,27,4060.74,100
chapter,260,421.69,100
lexeme,3038,36.09,100
verse,7957,13.78,100
word,109640,1.00,100


Acts


## Create a list of all verses for training

In [485]:
train_data_dict = ot_train_book_verses.copy()
train_data_dict.update(nt_train_book_verses)

ot_train_verses = [vrs for verses in ot_train_book_verses.values() for vrs in verses]
nt_train_verses = [vrs for verses in nt_train_book_verses.values() for vrs in verses]

print(len(ot_train_verses))
print(len(nt_train_verses))

train_verse_txts = ot_train_verses.copy()
train_verse_txts.extend(nt_train_verses)

train_verse_labels = [0 for i in range(len(ot_train_verses))]
train_verse_labels.extend([1 for i in range(len(nt_train_verses))])
print(len(train_verse_labels))

2115
3726
5841


# Generate Bag-of-Words counters & define the model's vocabulary

In [486]:
from collections.abc import Callable

In [487]:
def count_n_grams(
        word_count: int,
        words: list[str],
        ngram_formatter: Callable,
        span: int=3
    ) -> (int, Counter):
    """Count n-grams in the verse_words, using nltk.util.ngrams().
    
    ngram_formatter: Callable
        This is a pre-processing function or method to apply to the text of each verse,
        before passing it to nltk.util.ngrams().
        
        The format of strings passed to ngrams() function determines if the function
        returns a list of character n-grams or that of word n-grams.

        If you pass a single string, it returns character n-grams, and if you pass
        a list of strings (= words), it returns word n-grams.
    """
    # Format the verse to feed into ngrams()
    formatted_inputs = ngram_formatter(words)
    
    # Generate and count ngrams
    n_grams = list(ngrams(formatted_inputs, span))
    n_gram_counts = Counter(n_grams)
    
    # Keep the ngram counters
    return (word_count+len(words), n_gram_counts)

In [488]:
def make_vocab(verses: list[tuple[str, list[str]]], ngram_formatter: Callable, span: int=3) -> tuple[Counter, list[Counter]]:
    """Construct the model's vocabulary by extracting n-grams from verses.

    ngram_formatter: Callable
        This is a pre-processing function or method to apply to the text of each verse.
        See count_n_grams for more details.
    """
    word_cnt = 0
    n_gram_vocabs = None
    n_gram_counters = []

    for verse in verses:
        word_cnt, n_gram_counter = count_n_grams(word_cnt, verse[1], ngram_formatter, span)

        n_gram_counters.append(n_gram_counter)
    
        if n_gram_vocabs is None:
            n_gram_vocabs = n_gram_counter.copy()
        else:
            n_gram_vocabs.update(n_gram_counter)

    print(f"Total words parsed: {word_cnt}")
    return (n_gram_vocabs, n_gram_counters)

In [489]:
def identity(input: type) -> type:
    return input

def make_word_n_gram_vocab(verses: list[tuple[str, list[str]]], span: int=3) -> tuple[Counter, list[Counter]]:
    return make_vocab(verses, identity, span)

def make_char_n_gram_vocab(verses: list[tuple[str, list[str]]], span: int=3) -> tuple[Counter, list[Counter]]:
    return make_vocab(verses, ' '.join, span)

In [490]:
char_n_gram_vocabs = make_char_n_gram_vocab(train_verse_txts)
print(f"Found {len(char_n_gram_vocabs[0])} independent n-grams from {len(char_n_gram_vocabs[1])} verses!")

Total words parsed: 78289
Found 7441 independent n-grams from 5841 verses!


In [491]:
print(char_n_gram_vocabs[0])
# print(len(char_n_gram_vocabs.keys()))

Counter({('W', 'N', ' '): 4922, ('J', 'N', ' '): 4797, ('>', ' ', 'D'): 4133, ('N', '>', ' '): 3406, ('>', ' ', 'W'): 3355, ('L', '>', ' '): 2895, ('T', '>', ' '): 2850, (' ', '>', 'N'): 2700, ('>', ' ', '>'): 2690, ('>', ' ', 'L'): 2483, (' ', 'H', 'W'): 2378, (' ', 'W', '>'): 2289, (' ', 'L', 'H'): 2242, ('M', 'N', ' '): 2202, (' ', 'M', 'N'): 2188, ('J', '>', ' '): 2128, ('>', 'M', 'R'): 2083, ('N', ' ', '>'): 1942, ('R', '>', ' '): 1902, (' ', 'D', '>'): 1788, ('>', ' ', 'M'): 1764, ('H', 'W', 'N'): 1731, ('H', 'J', ' '): 1669, (' ', 'L', '>'): 1659, (' ', 'D', 'J'): 1656, ('N', ' ', 'D'): 1626, ('M', '>', ' '): 1602, ('>', ' ', 'H'): 1601, ('L', 'H', ' '): 1553, ('D', 'J', 'N'): 1541, ('W', 'H', 'J'): 1539, ('W', '>', 'M'): 1444, ('N', ' ', 'L'): 1435, ('M', 'R', ' '): 1416, ('N', ' ', 'W'): 1399, ('W', '>', ' '): 1321, (' ', '>', 'J'): 1319, ('>', ' ', 'B'): 1314, ('H', 'W', '>'): 1251, ('R', ' ', 'L'): 1244, ('>', 'N', 'T'): 1199, ('J', 'T', ' '): 1180, ('C', '>', ' '): 1174, ('

# Vectorise the verses

In [492]:
# Count the number of each n-gram per verse,
# reusing the counters created while constructing the vocabulary

assert(len(char_n_gram_vocabs[1]) == len(train_verse_txts))

In [493]:
def make_bow(vocabulary: dict) -> dict[tuple[str], int]:
    return dict.fromkeys(vocabulary.keys(), 0)

In [494]:
def make_feature(n_gram_vocabs: tuple[Counter, list[Counter]]) ->list[list[int]]:
    feature_list = []
    for i in range(len(n_gram_vocabs[1])):
        # Make a skeleton dict to use as the BoW feature, set all counts to 0
        n_gram_bow = make_bow(n_gram_vocabs[0])
        # Add counts to words that ecist in this verse
        n_gram_bow.update(n_gram_vocabs[1][i])
        verse_features = list(n_gram_bow.values())
        feature_list.append(verse_features)
    return feature_list

In [495]:
char_n_gram_feat = make_feature(char_n_gram_vocabs)

In [496]:
print(len(char_n_gram_feat))

5841


## Train classifier

In [497]:
# n_gram_train, n_gram_test, n_gram_train_y, n_gram_test_y = train_test_split(n_gram_feats, labels, test_size=0.2, random_state=0)

gnb = GaussianNB()

c_clf = gnb.fit(char_n_gram_feat, train_verse_labels)

print(f"{train_verse_labels[0]}, {train_verse_labels[-1]}")

0, 1


In [498]:
mnb = MultinomialNB()

c_mnb = mnb.fit(char_n_gram_feat, train_verse_labels)

print(f"{train_verse_labels[0]}, {train_verse_labels[-1]}")

0, 1


In [499]:
cnb = ComplementNB()

c_cnb = cnb.fit(char_n_gram_feat, train_verse_labels)

print(f"{train_verse_labels[0]}, {train_verse_labels[-1]}")

0, 1


## Prepare test data

In [500]:
def merge_counts(counter: Counter, feat_vec: dict) -> None:
    """Update the value of feat_vec with the value found in Counter.
    
    This function makes sure that the length of the feature vector
    doesn't change when updating the n-gram counts in feat_vec.
    If you use dict.update(), you accidentally add extra vocabs to
    feat_vec (which changes the length of the feature vector).
    """
    for n_gram in counter.keys():
        # print(n_gram)
        if n_gram in feat_vec.keys():
            feat_vec[n_gram] = counter[n_gram]

In [501]:
def vectorise(
        verses: list[tuple[str, list[str]]],
        bag_of_words: Counter,
        ngram_formatter: Callable,
        span: int=3
    ) -> list[list[int]]:
    """Count n-grams in the verses, using the vocabulary from bag_of_words."""
    # Get the verse of format [(VERSE_REF, [VERSE_translit_WORDS], [VERSE_syriac_WORDS])]
    # Generate two lists: [[N_GRAM_COUNTS] per each verse] & [LABEL per each verse]
    labels = []
    wc = 0
    verse_n_grams = []
    for verse in verses:
        wc, local_n_gram_counts = count_n_grams(word_count=wc, words=verse[1], ngram_formatter=ngram_formatter, span=3)
        n_gram_bow = dict.fromkeys(bag_of_words.keys(), 0)
        merge_counts(local_n_gram_counts, n_gram_bow)
        verse_n_grams.append(list(n_gram_bow.values()))
    print(f"Parsed {wc} words from {len(verses)} verses")
    return verse_n_grams

### OT

In [502]:
ot_test_verses = [vrs for book in ot_test_book_verses.values() for vrs in book]

# Labels
ot_test_y_np = np.empty(len(ot_test_verses), dtype=int)
ot_test_y_np.fill(0)
print(ot_test_y_np.shape)

# Vectorise with Bag-of-Words counting
ot_test_X = vectorise(ot_test_verses, make_bow(char_n_gram_vocabs[0]), ' '.join)
ot_test_X_np = np.array(ot_test_X)

(555,)
Parsed 8348 words from 555 verses


### NT

In [503]:
nt_test_verses = [vrs for book in nt_test_book_verses.values() for vrs in book]

# Labels
nt_test_y_np = np.empty(len(nt_test_verses), dtype=int)
nt_test_y_np.fill(1)
print(nt_test_y_np.shape)

# Features
nt_test_X = vectorise(nt_test_verses, make_bow(char_n_gram_vocabs[0]), ' '.join)
nt_test_X_np = np.array(nt_test_X)

(1007,)
Parsed 15383 words from 1007 verses


In [504]:
all_test_y = []
all_test_y.extend(ot_test_y_np)
all_test_y.extend(nt_test_y_np)

## Evaluate

In [505]:
def predict(
        classifier, test_X: np.ndarray, test_labels: np.array
    ) -> (np.array, np.array, np.array):
    """Predict on the data with a given classifier, and return some simple statistics.

    The classifier must have a method `.predict()`.
    """
    y_pred = classifier.predict(test_X)
    
    num_samples = test_X.shape[0]
    num_mislabels = ((test_labels != y_pred).sum())
    num_correct = num_samples - num_mislabels
    
    print("Number of mislabeled points out of a total %d verses: %d" % (num_samples, num_mislabels))
    print(f"Local accuracy: {accuracy_score(test_labels, y_pred):.02f}")

    return y_pred, num_mislabels, num_correct

In [506]:
def convert(probas: np.array):
    result = np.empty(0, dtype=int)
    """Convert list of probabilities to a list of labels."""
    for i in range(len(probas)):
        probability = probas[i]
        if probability[0] > 0.5:
            result = np.append(result, 0)
        elif probability[1] > 0.5:
            result = np.append(result, 1)
        elif probability[0] == probability[1]:
            result = np.append(result, None)
        else:
            raise ValueError(f"Something is wrong with the probability at index: {i}!")
    return result


def predict_proba(
        classifier: type, test_X: np.ndarray, test_labels: np.array
    ) -> (np.array, np.array, np.array, np.array):
    """Predict on the data with the classifier, and return some simple statistics.

    The classifier must have a method `.predict_proba()`.
    """
    y_pred_proba = classifier.predict_proba(test_X)
    y_pred = convert(y_pred_proba)
    
    num_samples = test_X.shape[0]
    num_mislabels = ((test_labels != y_pred).sum())
    num_correct = num_samples - num_mislabels
    
    print("Number of mislabeled points out of a total %d verses: %d" % (num_samples, num_mislabels))
    print(f"Local accuracy: {accuracy_score(test_labels, y_pred):.02f}")
    return y_pred_proba, y_pred, num_mislabels, num_correct

In [507]:
def metricise(y_true: list[int], y_pred_pos: list[int], y_pred_neg):
    """Measure the performance of a classifier using the outputs."""
    y_pred = []
    y_pred.extend(y_pred_pos)
    y_pred.extend(y_pred_neg)
    y_pred = np.array(y_pred)
    
    accuracy = accuracy_score(y_true, y_pred)
    recall = recall_score(y_true, y_pred)
    f1_result = f1_score(y_true, y_pred)
    
    print(f"Overall Accuracy: {precision:.02f}")
    print(f"Overall Recall: {recall:.02f}")
    print(f"F1 Score: {f1_result:.02f}")
    return accuracy, recall, f1_result

In [508]:
ot_y_proba, ot_y_pred, num_fp, num_tn = predict_proba(c_mnb, ot_test_X_np, ot_test_y_np)

Number of mislabeled points out of a total 555 verses: 96
Local accuracy: 0.83


In [509]:
nt_y_proba, nt_y_pred, num_fn, num_tp = predict_proba(c_mnb, nt_test_X_np, nt_test_y_np)

Number of mislabeled points out of a total 1007 verses: 23
Local accuracy: 0.98


In [510]:
metricise(all_test_y, ot_y_pred, nt_y_pred)

Overall Accuracy: 0.92
Overall Recall: 0.98
F1 Score: 0.94


(0.9238156209987196, 0.9771598808341608, 0.9429803545759463)

## Format data for manual inspection

In [511]:
def csvify(key: str, verses_d: dict[str, list[tuple[str, list[str]]]]) -> str:
    """Convert the verse data into a CSV-formatted string."""
    # Set header line
    result = '"Verse Reference No.","Probability","No. Words","ܐܠܦܒܝܬ ܣܘܪܝܝܐ","ETCBC Transliteration"\n'
    # Extract & format verse data
    verses = verses_d[key]
    for verse in verses:
        line = f'"{verse[0]}",{verse[1]:.02f},{len(verse[2])},"{' '.join(verse[2])}","{' '.join(verse[3])}"\n'
        result += line
    return result

In [515]:
ot_mislabels = {
    "Deuteronomy": []
}

nt_mislabels = {
    "Acts": []
}

pos = 0

# OT
for i in range(len(ot_test_verses)):
    if ot_test_y_np[i] != ot_y_pred[i]:
        res = (
            ot_test_verses[i][0],
            ot_y_proba[i][0],
            ot_test_verses[i][2],
            ot_test_verses[i][1]
        )
        ot_mislabels["Deuteronomy"].append(res)


# NT
for i in range(len(nt_y_pred)):
    if nt_test_y_np[i] != nt_y_pred[i]:
        res = (
            nt_test_verses[i][0],
            nt_y_proba[i][1],
            nt_test_verses[i][2],
            nt_test_verses[i][1]
        )
        nt_mislabels["Acts"].append(res)

In [516]:
print(len(ot_mislabels["Deuteronomy"]))

96


In [517]:
Path(f"./out/prediction_data_ot.csv").write_text(csvify("Deuteronomy", ot_mislabels))
Path(f"./out/prediction_data_nt.csv").write_text(csvify("Acts", nt_mislabels))

3853

## Try different n of character n-grams

In [518]:
for char_n in range(1,6):
    print(f"\nchar_n = {char_n}")
    char_n_gram_vocabs = make_char_n_gram_vocab(train_verse_txts, span=char_n)
    print(f"Found {len(char_n_gram_vocabs[0])} independent n-grams from {len(char_n_gram_vocabs[1])} verses!")

    if char_n == 2 or char_n == 4:
        print(list(char_n_gram_vocabs[0].keys())[0:11])
    
    char_n_gram_feat = make_feature(char_n_gram_vocabs)
    
    mnb = MultinomialNB()
    c_mnb = mnb.fit(char_n_gram_feat, train_verse_labels)
    
    print("--> OT")
    ot_test_X_c = vectorise(ot_test_verses, make_bow(char_n_gram_vocabs[0]), ' '.join, span=char_n)
    ot_test_X_c_np = np.array(ot_test_X_c)
    ot_y_proba, ot_y_pred, num_fn, num_tp = predict_proba(c_mnb, ot_test_X_c_np, ot_test_y_np)
    
    print("--> NT")
    nt_test_X_c = vectorise(nt_test_verses, make_bow(char_n_gram_vocabs[0]), ' '.join, span=char_n)
    nt_test_X_c_np = np.array(nt_test_X_c)
    nt_y_proba, nt_y_pred, num_fn, num_tp = predict_proba(c_mnb, nt_test_X_c_np, nt_test_y_np)
    
    print("--> ALL")
    metricise(all_test_y, ot_y_pred, nt_y_pred)


char_n = 1
Total words parsed: 78289
Found 27 independent n-grams from 5841 verses!
--> OT
Parsed 8348 words from 555 verses
Number of mislabeled points out of a total 555 verses: 555
Local accuracy: 0.00
--> NT
Parsed 15383 words from 1007 verses
Number of mislabeled points out of a total 1007 verses: 0
Local accuracy: 1.00
--> ALL
Overall Accuracy: 0.92
Overall Recall: 1.00
F1 Score: 0.78

char_n = 2
Total words parsed: 78289
Found 641 independent n-grams from 5841 verses!
[('B', 'R'), ('R', 'C'), ('C', 'J'), ('J', 'T'), ('T', ' '), (' ', 'B'), ('R', '>'), ('>', ' '), (' ', '>'), ('>', 'L'), ('L', 'H')]
--> OT
Parsed 8348 words from 555 verses
Number of mislabeled points out of a total 555 verses: 555
Local accuracy: 0.00
--> NT
Parsed 15383 words from 1007 verses
Number of mislabeled points out of a total 1007 verses: 0
Local accuracy: 1.00
--> ALL
Overall Accuracy: 0.92
Overall Recall: 1.00
F1 Score: 0.78

char_n = 3
Total words parsed: 78289
Found 7441 independent n-grams from 58

## Word n-grams with various values for n

In [519]:
word_n = 1  # n of word n-grams

In [520]:
word_n_gram_vocabs = make_word_n_gram_vocab(train_verse_txts, span=word_n)
print(f"Found {len(word_n_gram_vocabs[0])} independent n-grams from {len(word_n_gram_vocabs[1])} verses!")

Total words parsed: 78289
Found 14178 independent n-grams from 5841 verses!


In [521]:
word_n_gram_feat = make_feature(word_n_gram_vocabs)
print(len(word_n_gram_feat[0]))
print(len(word_n_gram_feat))

14178
5841


In [522]:
mnb = MultinomialNB()

w_mnb = mnb.fit(word_n_gram_feat, train_verse_labels)

print(f"{train_verse_labels[0]}, {train_verse_labels[-1]}")

0, 1


In [523]:
# Vectorise with Bag-of-Words counting
ot_test_X_w = vectorise(ot_test_verses, make_bow(word_n_gram_vocabs[0]), identity, span=word_n)
ot_test_X_w_np = np.array(ot_test_X_w)

Parsed 8348 words from 555 verses


In [524]:
ot_y_proba, ot_y_pred, num_fn, num_tp = predict_proba(w_mnb, ot_test_X_w_np, ot_test_y_np)

Number of mislabeled points out of a total 555 verses: 555
Local accuracy: 0.00


In [525]:
# Features
nt_test_X_w = vectorise(nt_test_verses, make_bow(word_n_gram_vocabs[0]), identity, span=word_n)
nt_test_X_w_np = np.array(nt_test_X_w)

Parsed 15383 words from 1007 verses


In [526]:
nt_y_proba, nt_y_pred, num_fn, num_tp = predict_proba(w_mnb, nt_test_X_w_np, nt_test_y_np)

Number of mislabeled points out of a total 1007 verses: 0
Local accuracy: 1.00


In [527]:
metricise(all_test_y, ot_y_pred, nt_y_pred)

Overall Accuracy: 0.92
Overall Recall: 1.00
F1 Score: 0.78


(0.6446862996158771, 1.0, 0.7839626313740755)

## Word n-grams with various values for n

In [528]:
for word_n in range(1,6):
    print(f"\nword_n = {word_n}")
    word_n_gram_vocabs = make_word_n_gram_vocab(train_verse_txts, span=word_n)
    print(f"Found {len(word_n_gram_vocabs[0])} independent n-grams from {len(word_n_gram_vocabs[1])} verses!")
    
    word_n_gram_feat = make_feature(word_n_gram_vocabs)
    
    mnb = MultinomialNB()
    w_mnb = mnb.fit(word_n_gram_feat, train_verse_labels)
    
    print("OT")
    ot_test_X_w = vectorise(ot_test_verses, make_bow(word_n_gram_vocabs[0]), identity, span=word_n)
    ot_test_X_w_np = np.array(ot_test_X_w)
    ot_y_proba, ot_y_pred, num_fn, num_tp = predict_proba(w_mnb, ot_test_X_w_np, ot_test_y_np)
    
    print("NT")
    nt_test_X_w = vectorise(nt_test_verses, make_bow(word_n_gram_vocabs[0]), identity, span=word_n)
    nt_test_X_w_np = np.array(nt_test_X_w)
    nt_y_proba, nt_y_pred, num_fn, num_tp = predict_proba(w_mnb, nt_test_X_w_np, nt_test_y_np)

    print("ALL")
    metricise(all_test_y, ot_y_pred, nt_y_pred)


word_n = 1
Total words parsed: 78289
Found 14178 independent n-grams from 5841 verses!
OT
Parsed 8348 words from 555 verses
Number of mislabeled points out of a total 555 verses: 555
Local accuracy: 0.00
NT
Parsed 15383 words from 1007 verses
Number of mislabeled points out of a total 1007 verses: 0
Local accuracy: 1.00
ALL
Overall Accuracy: 0.92
Overall Recall: 1.00
F1 Score: 0.78

word_n = 2
Total words parsed: 78289
Found 49878 independent n-grams from 5841 verses!
OT
Parsed 8348 words from 555 verses
Number of mislabeled points out of a total 555 verses: 555
Local accuracy: 0.00
NT
Parsed 15383 words from 1007 verses
Number of mislabeled points out of a total 1007 verses: 0
Local accuracy: 1.00
ALL
Overall Accuracy: 0.92
Overall Recall: 1.00
F1 Score: 0.78

word_n = 3
Total words parsed: 78289
Found 58968 independent n-grams from 5841 verses!
OT
Parsed 8348 words from 555 verses
Number of mislabeled points out of a total 555 verses: 457
Local accuracy: 0.18
NT
Parsed 15383 words f

## SVM

### Character N-grams

In [416]:
svc = SVC()

csvc_clf = svc.fit(char_n_gram_feat, train_verse_labels)

# OT
ot_y_pred, num_fp, num_tn = predict(csvc_clf, ot_test_X_np, ot_test_y_np)

# NT
nt_y_pred, num_fn, num_tp = predict(csvc_clf, nt_test_X_np, nt_test_y_np)

metricise(all_test_y, ot_y_pred, nt_y_pred)

Number of mislabeled points out of a total 555 OT verses: 142
Local accuracy: 0.74
Number of mislabeled points out of a total 1007 OT verses: 23
Local accuracy: 0.98
Overall Precision: 0.87
Overall Recall: 0.98
F1 Score: 0.92


### Word N-grams

In [422]:
import time

svc = SVC()

print(time.asctime())
wsvc_clf = svc.fit(word_n_gram_feat, train_verse_labels)

print(time.asctime())
print("fitted!")

Sat Mar 22 14:16:19 2025
Sat Mar 22 14:21:19 2025
fitted!


In [423]:
# OT
ot_y_pred, num_fp, num_tn = predict(wsvc_clf, ot_test_X_w_np, ot_test_y_np)

# NT
nt_y_pred, num_fn, num_tp = predict(wsvc_clf, nt_test_X_w_np, nt_test_y_np)

metricise(all_test_y, ot_y_pred, nt_y_pred)

Number of mislabeled points out of a total 555 OT verses: 548
Local accuracy: 0.01
Number of mislabeled points out of a total 1007 OT verses: 0
Local accuracy: 1.00
Overall Precision: 0.65
Overall Recall: 1.00
F1 Score: 0.79


## Word counts in the dataset

In [425]:
# Average no. words (with std) per verse, in training and test sets, also by books (possibly).

all_wc = []
ot_wc = []
nt_wc = []
all_cc = []
nt_cc = []
ot_cc = []

for book in ot_train_book_verses.keys():
    print(f"Book: {book}")
    book_wc = []
    book_cc = []
    verses = ot_train_book_verses[book]
    for ot_verse in verses:
        # print(ot_verse)
        wc = len(ot_verse[1])
        c_cnt = [len(word) for word in ot_verse]
        ot_wc.append(wc)
        all_wc.append(wc)
        book_wc.append(wc)
        all_cc.extend(c_cnt)
        ot_cc.extend(c_cnt)
        book_cc.extend(c_cnt)
    print(f"Number of verses: {len(book_wc)}")
    print(f"Avg word count: {np.average(book_wc)}")
    print(f"Word count std: {np.std(book_wc)}\n")
    print(f"Avg char count: {np.average(book_cc)}")
    print(f"Char count std: {np.std(book_cc)}")

for book in nt_train_book_verses.keys():
    print(f"Book: {book}")
    book_wc = []
    book_cc = []
    verses = nt_train_book_verses[book]
    for nt_verse in verses:
        wc = len(nt_verse[1])
        c_cnt = [len(word) for word in ot_verse]
        nt_wc.append(wc)
        all_wc.append(wc)
        book_wc.append(wc)
        all_cc.extend(c_cnt)
        nt_cc.extend(c_cnt)
        book_cc.extend(c_cnt)
    print(f"Number of verses: {len(book_wc)}")
    print(f"Avg word count: {np.average(book_wc)}")
    print(f"Word count std: {np.std(book_wc)}\n")
    print(f"Avg char count: {np.average(book_cc)}")
    print(f"Char count std: {np.std(book_cc)}")

Book: Genesis
Number of verses: 1533
Avg word count: 13.09849967384214
Word count std: 4.8717489401625675

Avg char count: 17.583387692976736
Char count std: 7.494576872678524
Book: Exodus
Number of verses: 582
Avg word count: 14.474226804123711
Word count std: 5.5310919676008075

Avg char count: 18.06758304696449
Char count std: 6.809972860104494
Book: Matthew
Number of verses: 1071
Avg word count: 13.052287581699346
Word count std: 4.985232336843567

Avg char count: 22.666666666666668
Char count std: 2.3570226039551585
Book: Mark
Number of verses: 678
Avg word count: 12.969026548672566
Word count std: 4.831428201103894

Avg char count: 22.666666666666668
Char count std: 2.3570226039551585
Book: Luke
Number of verses: 1098
Avg word count: 13.301457194899818
Word count std: 4.995919798628114

Avg char count: 22.666666666666668
Char count std: 2.357022603955158
Book: John
Number of verses: 879
Avg word count: 14.1160409556314
Word count std: 5.495283304087178

Avg char count: 22.6666666

In [426]:
# Overall train verses statistics
print(f"Avg word count across all train verses: {np.average(all_wc)}\n")

# OT train verses statistics
print(f"OT train verses: {len(ot_wc)}")
print(f"Avg word count: {np.average(ot_wc)}")
print(f"Avg char count: {np.average(ot_cc)}")
print(f"Word count std: {np.std(ot_wc)}")
print(f"Char count std: {np.std(ot_cc)}\n")

# NT train verses statistics
print(f"NT train verses: {len(nt_wc)}")
print(f"Avg word count: {np.average(nt_wc)}")
print(f"Avg char count: {np.average(nt_cc)}")
print(f"Word count std: {np.std(nt_wc)}")
print(f"Char count std: {np.std(nt_cc)}\n")

Avg word count across all train verses: 13.403355589796268

OT train verses: 2115
Avg word count: 13.477068557919623
Avg char count: 17.716627265563435
Word count std: 5.098909994539582
Char count std: 7.3157805817189105

NT train verses: 3726
Avg word count: 13.361513687600644
Avg char count: 22.666666666666668
Word count std: 5.105017399286848
Char count std: 2.3570226039551585



In [428]:
all_test_wc = []
ot_test_wc = []
nt_test_wc = []

for book in ot_test_book_verses.keys():
    print(f"Book: {book}")
    book_test_wc = []
    verses = ot_test_book_verses[book]
    for ot_verse in verses:
        # print(ot_verse)
        wc = len(ot_verse[1])
        # print(wc)
        ot_test_wc.append(wc)
        all_test_wc.append(wc)
        book_test_wc.append(wc)
    print(f"Number of verses: {len(book_test_wc)}")
    print(f"Avg word count: {np.average(book_test_wc)}")
    print(f"Std: {np.std(book_test_wc)}\n")

for book in nt_test_book_verses.keys():
    print(f"Book: {book}")
    book_test_wc = []
    verses = nt_test_book_verses[book]
    for nt_verse in verses:
        wc = len(nt_verse[1])
        nt_test_wc.append(wc)
        all_test_wc.append(wc)
        book_test_wc.append(wc)
    print(f"Number of verses: {len(book_test_wc)}")
    print(f"Avg word count: {np.average(book_test_wc)}")
    print(f"Std: {np.std(book_test_wc)}\n")

# Overall test verses statistics
print(f"Avg word count across all words: {np.average(all_wc)}\n")

Book: Deuteronomy
Number of verses: 555
Avg word count: 15.04144144144144
Std: 5.780371702697472

Book: Acts
Number of verses: 1007
Avg word count: 15.276067527308838
Std: 5.572600478658499

Avg word count across all words: 13.403355589796268

